In [1]:
pip install nltk


Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import re
import string

from nltk.corpus import stopwords
import nltk

from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [3]:
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/aximsoft/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
df = pd.read_csv(
    "IMDB Dataset.csv"
)

df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
print("Dataset Shape:", df.shape)

print("\nColumns:")
print(df.columns)

print("\nSentiment Distribution:")
print(df["sentiment"].value_counts())

df.head()

Dataset Shape: (50000, 2)

Columns:
Index(['review', 'sentiment'], dtype='str')

Sentiment Distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [6]:
df["review"] = df["review"].astype(str).str.lower()

df[["review", "sentiment"]].head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


In [7]:
def remove_html(text):

    clean = re.compile("<.*?>")

    return re.sub(clean, " ", text)


df["clean_review"] = df["review"].apply(remove_html)

df[["review", "clean_review"]].head()

,review,clean_review
0,one of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...
1,a wonderful little production. <br /><br />the...,a wonderful little production. the filming t...
2,i thought this was a wonderful way to spend ti...,i thought this was a wonderful way to spend ti...
3,basically there's a family where a little boy ...,basically there's a family where a little boy ...
4,"petter mattei's ""love in the time of money"" is...","petter mattei's ""love in the time of money"" is..."


In [8]:
def remove_special_characters(text):

    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()


df["clean_review"] = df["clean_review"].apply(
    remove_special_characters
)

df[["clean_review", "sentiment"]].head()

,clean_review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production the filming tech...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there s a family where a little boy ...,negative
4,petter mattei s love in the time of money is a...,positive


In [9]:
stop_words = set(stopwords.words("english"))
negation_words = {
    "no",
    "not",
    "never",
    "neither",
    "nor"
}
stop_words_without_negation = (
    stop_words - negation_words
)
def remove_stopwords(text):
    words = text.split()
    filtered_words = [
        word
        for word in words
        if word not in stop_words_without_negation
    ]
    return " ".join(filtered_words)

In [10]:
print("Original Review:\n")

print(df["review"].iloc[0])

print("\n" + "=" * 80 + "\n")

print("Cleaned Review:\n")

print(df["clean_review"].iloc[0])

Original Review:

one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked. they are right, as this is exactly what happened with me.<br /><br />the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go. trust me, this is not a show for the faint hearted or timid. this show pulls no punches with regards to drugs, sex or violence. its is hardcore, in the classic use of the word.<br /><br />it is called oz as that is the nickname given to the oswald maximum security state penitentary. it focuses mainly on emerald city, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. em city is home to many..aryans, muslims, gangstas, latinos, christians, italians, irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />i would say the main appeal of the show

In [11]:
# Encode Labels
df["label"] = df["sentiment"].map(
    {
        "negative": 0,
        "positive": 1
    }
)

df[["sentiment", "label"]].head()

,sentiment,label
0,positive,1
1,positive,1
2,positive,1
3,negative,0
4,positive,1


In [12]:
print(df["label"].value_counts())

label
1    25000
0    25000
Name: count, dtype: int64


In [13]:
X = df["clean_review"]

y = df["label"]


X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42,
    stratify=y
)


X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.1765,
    random_state=42,
    stratify=y_temp
)

In [14]:
print("Training Data:", X_train.shape)
print("Validation Data:", X_val.shape)
print("Test Data:", X_test.shape)

Training Data: (34998,)
Validation Data: (7502,)
Test Data: (7500,)


In [15]:
VOCAB_SIZE = 20000


tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token="<OOV>"
)


tokenizer.fit_on_texts(X_train)

In [16]:
print(
    "Number of Words in Vocabulary:",
    len(tokenizer.word_index)
)

Number of Words in Vocabulary: 85744


In [17]:
X_train_sequences = tokenizer.texts_to_sequences(
    X_train
)

X_val_sequences = tokenizer.texts_to_sequences(
    X_val
)

X_test_sequences = tokenizer.texts_to_sequences(
    X_test
)

In [18]:
print("Original Review:")

print(X_train.iloc[0])

print("\nTokenized Sequence:")

print(X_train_sequences[0])

Original Review:
now i love bad old skifee movies as much as most people and i understand that a budget is a budget that said planet of the dinosaurs is as bad as a bad movie can get the thing has no actors and only one attractive female whom they kill off two minutes after swimming ashore there are literally no redeeming qualities to be found in this pile of wasted celluloid the only thing not wasted was paper the screenplay must have been no more than four pages long surely no one actually wrote dialogue this pointless i m constantly amazed that such movies ever got made much less released i m only glad i didn t pay to see this waste of time it s minutes of my life i ll never get back

Tokenized Sequence:
[148, 10, 114, 75, 159, 1, 99, 15, 73, 15, 90, 81, 4, 10, 378, 12, 3, 328, 7, 3, 328, 12, 307, 1281, 5, 2, 3730, 7, 15, 75, 15, 3, 75, 18, 51, 78, 2, 150, 47, 57, 151, 4, 66, 29, 1535, 638, 908, 33, 495, 123, 107, 228, 101, 4261, 13904, 40, 26, 1241, 57, 1721, 2326, 6, 30, 253, 9, 1

In [19]:
MAX_LENGTH = 200


X_train_padded = pad_sequences(
    X_train_sequences,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)


X_val_padded = pad_sequences(
    X_val_sequences,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)


X_test_padded = pad_sequences(
    X_test_sequences,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

In [20]:
print("Training Shape:", X_train_padded.shape)

print("Validation Shape:", X_val_padded.shape)

print("Test Shape:", X_test_padded.shape)

Training Shape: (34998, 200)
Validation Shape: (7502, 200)
Test Shape: (7500, 200)


In [21]:
import pickle
import os

os.makedirs("processed_data", exist_ok=True)

In [22]:
with open(
    "processed_data/tokenizer.pkl",
    "wb"
) as file:

    pickle.dump(tokenizer, file)

In [23]:
np.save(
    "processed_data/X_train.npy",
    X_train_padded
)

np.save(
    "processed_data/X_val.npy",
    X_val_padded
)

np.save(
    "processed_data/X_test.npy",
    X_test_padded
)


np.save(
    "processed_data/y_train.npy",
    y_train
)

np.save(
    "processed_data/y_val.npy",
    y_val
)

np.save(
    "processed_data/y_test.npy",
    y_test
)

In [26]:
df.to_csv("processed_data/imdb_cleaned.csv",index=False)

In [27]:
print("Phase 2 preprocessing completed successfully!")

print("\nFinal Data Shapes:")

print("X_train:", X_train_padded.shape)
print("X_val:", X_val_padded.shape)
print("X_test:", X_test_padded.shape)

print("\nVocabulary Size:", VOCAB_SIZE)
print("Maximum Sequence Length:", MAX_LENGTH)

Phase 2 preprocessing completed successfully!

Final Data Shapes:
X_train: (34998, 200)
X_val: (7502, 200)
X_test: (7500, 200)

Vocabulary Size: 20000
Maximum Sequence Length: 200
